In [ ]:
from dotenv import load_dotenv
load_dotenv()

import uuid
from typing import List
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

In [ ]:
store=InMemoryStore()

In [ ]:
extractor_llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
class MemoryDecision(BaseModel):
    should_write: bool = Field(
        description="Whether to store any memories"
    )
    memories: List[str] = Field(
        description="Atomic user memories to store",
        default_factory=list
    )

In [ ]:
memory_extractor=extractor_llm.with_structured_output(MemoryDecision)

In [ ]:
def remember_only_node(
    state: MessagesState,
    config: RunnableConfig,
    store: BaseStore
):
    user_id = config["configurable"]["user_id"]
    namespace = ("user", user_id, "details")

    last_message = state["messages"][-1].content

    decision: MemoryDecision = memory_extractor.invoke(
        [
            SystemMessage(
                content="""
                Identify important information about the user
                that should be remembered for future conversations.

                Extract memories as short, atomic statements.

                Only store information that is useful for future
                conversations, such as:
                - User preferences
                - User interests
                - User goals
                - User background
                - User projects
                """
            ),
            {
                "role": "user",
                "content": last_message
            }
        ]
    )

    if decision.should_write:
        for memory in decision.memories:
            memory_id = str(uuid.uuid4())

            store.put(
                namespace,
                memory_id,
                {
                    "data": memory
                }
            )

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("remember", remember_only_node)

builder.add_edge(START, "remember")
builder.add_edge("remember", END)

graph = builder.compile()

In [ ]:
config = {
    "configurable": {
        "user_id": "u1"
    }
}

result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I prefer concise answers and I am currently learning LangGraph."
            }
        ]
    },
    config
)

print(result)

In [ ]:
items = store.search(("user", "u1", "details"))

In [ ]:
for item in items:
    print(item.value)

Without Duplicate


In [ ]:
def remember_only_node(
    state: MessagesState,
    config: RunnableConfig,
    store: BaseStore
):
    user_id = config["configurable"]["user_id"]
    namespace = ("user", user_id, "details")

    last_message = state["messages"][-1].content

    decision: MemoryDecision = memory_extractor.invoke(
        [
            SystemMessage(
                content="""
                Identify important information about the user
                that should be remembered for future conversations.

                Extract memories as short, atomic statements.
                Do not extract information that is not useful
                for future conversations.
                """
            ),
            {
                "role": "user",
                "content": last_message
            }
        ]
    )

    if decision.should_write:
        existing_items = store.search(namespace)

        existing_memories = {
            item.value.get("data", "").strip().lower()
            for item in existing_items
        }

        for memory in decision.memories:
            memory = memory.strip()

            if memory.lower() not in existing_memories:
                memory_id = str(uuid.uuid4())

                store.put(
                    namespace,
                    memory_id,
                    {"data": memory}
                )